# Notebook 21 v4 — Memory re-measurement (hybrid direct-vision + text-only-language)

**Author:** Sunita Sahu · **Phase:** 3 (additive to Phase-2 master report)

**Cluster B.2 of the Phase-3 task. POD REQUIRED.**

## Where v3 failed

v3 tried to use the model's own `AutoProcessor` to build unified full-model inputs, then run one forward with hooks. Both steps failed on Molmo:
* **Processor**: `ImportError — Molmo's processor requires tensorflow` (a heavy dep we'd rather not install just for image preprocessing).
* **Fallback kwargs**: `MolmoForCausalLM.forward() got an unexpected keyword argument 'pixel_values'` — Molmo's *model wrapper* uses `images=`, not the standard `pixel_values=`.

So all 3 forward attempts raised TypeError → no per-component hook fired → `total_mb = None` → nothing was patched into `nb04_memory_summary.csv`.

## v4 strategy — two independent paths

| Component | Method | Why it works |
|---|---|---|
| **Vision** | Direct call: `vision_backbone(**kwargs)` with a kwarg sweep (`pixel_values`/`images`, 4D/5D shapes, positional) | The vision backbones in Molmo (`OLMoPretrainedVisionBackbone`) and DeepSeek-VL2 (`SigLIP-SO400M`) both have their own `forward()`. Direct call bypasses the model wrapper. |
| **Language** | `model(input_ids=..., attention_mask=...)` text-only + hook all language descendants | Text-only forward routes through the LLM body without needing tensorflow or `images` kwargs. Hooks on the leaf transformer modules capture peak. |
| **Total** | `vision + language` (sum); when one is None, report the one we got | Always populated as long as either path produced a number |

> **CI reporting standard (§4.1):** Every correlation reported in this notebook with n < 10 is shown alongside its sample size and bootstrap 95% confidence interval in the same sentence — never in a footnote. Tag: **exploratory** unless explicitly upgraded to confirmatory.

---
## 1. Pod setup + GPU check

In [1]:
# %%capture
# !pip install -q -r ../requirements.txt

In [2]:
import os, sys, gc, json, time, shutil, math
from pathlib import Path
import numpy as np, pandas as pd, torch

if os.environ.get('HF_HUB_ENABLE_HF_TRANSFER') == '1':
    try:
        import hf_transfer
        print('✓ hf_transfer available')
    except ImportError:
        os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

NB_DIR = Path.cwd()
CODE_ACTIVE = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
sys.path.insert(0, str(CODE_ACTIVE))

MEASUREMENT_DIR = CODE_ACTIVE / 'outputs' / 'measurements'
FIG_DIR = CODE_ACTIVE / 'outputs' / 'figures' / 'phase3'
PAPER_DIR = CODE_ACTIVE / 'outputs' / 'paper_track'
for d in (FIG_DIR, PAPER_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), 'nb21 requires CUDA'
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name}  ({p.total_memory/1e9:.1f} GB)')
print(f'torch: {torch.__version__}')

✓ hf_transfer available
GPU: NVIDIA A100-SXM4-80GB  (85.1 GB)
torch: 2.8.0+cu128


---
## 2. v4 helpers — direct vision call + text-only language forward

In [3]:
from vlm_audit import MODELS, extract_components
from vlm_audit.runner import _load_model

def reset_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def find_callable_descendants(component, max_depth=4):
    """DFS; return list of (path, module) with own forward (not nn.Module._forward_unimplemented)."""
    results = []
    def _r(mod, path, depth):
        if depth > max_depth: return
        if type(mod).forward is not torch.nn.Module.forward:
            results.append((path or '<self>', mod))
            return
        for name, child in mod.named_children():
            _r(child, f'{path}.{name}' if path else name, depth+1)
    _r(component, '', 0)
    return results

def measure_vision_direct(vision_comp, image_hw=(336, 336), device='cuda', repeats=3):
    """Direct call on the vision module with a kwarg sweep. Returns (peak_MB, working_pattern)."""
    H, W = image_hw
    pix4 = torch.randn(1, 3, H, W, dtype=torch.float16, device=device)
    pix5 = pix4.unsqueeze(1)  # (1, 1, 3, H, W) — single-crop multi-crop wrap, Molmo style
    # Molmo image_masks (1, num_crops, H, W) of 1s; image_input_idx zeros
    img_mask = torch.ones(1, 1, H, W, dtype=torch.float16, device=device)
    candidates = [
        ('pixel_values_4d',           {'pixel_values': pix4}),
        ('images_4d',                 {'images': pix4}),
        ('images_5d',                 {'images': pix5}),
        ('images_5d_with_mask',       {'images': pix5, 'image_masks': img_mask}),
        ('image_4d',                  {'image': pix4}),
    ]
    last_err = None
    for name, kw in candidates:
        peaks = []
        try:
            for i in range(repeats):
                reset_cuda()
                before = torch.cuda.memory_allocated()
                with torch.no_grad():
                    _ = vision_comp(**kw)
                torch.cuda.synchronize()
                peak = (torch.cuda.max_memory_allocated() - before) / 1024**2
                peaks.append(peak)
            return float(np.median(peaks)), name
        except (TypeError, RuntimeError, AttributeError) as e:
            last_err = f'{name}: {type(e).__name__}: {str(e)[:60]}'
            continue
        except torch.cuda.OutOfMemoryError as e:
            print(f'    [vision] OOM on {name}; skipping vision direct-call entirely')
            reset_cuda()
            return None, f'OOM_{name}'
    print(f'    [vision] no kwarg pattern worked; last error: {last_err}')
    return None, None

def measure_language_textonly(model, language_comp, text_tokens=32, device='cuda', repeats=3):
    """Call model(input_ids=..., attention_mask=...) text-only; hook language descendants.
    Returns (peak_MB, n_descendants_fired)."""
    ids = torch.randint(0, 30000, (1, text_tokens), device=device)
    mask = torch.ones_like(ids)
    descs = find_callable_descendants(language_comp)
    if not descs:
        # if no descendants, still hook the container itself — might fire via __call__
        descs = [('<self>', language_comp)]

    state = {}
    handles = []
    for path, dmod in descs:
        def mk_pre(mid):
            def pre(module, args):
                torch.cuda.synchronize()
                s = state.setdefault(mid, {'min_b': math.inf, 'max_a': -math.inf, 'fired': False})
                s['min_b'] = min(s['min_b'], torch.cuda.memory_allocated())
                s['fired'] = True
            return pre
        def mk_post(mid):
            def post(module, args, output):
                torch.cuda.synchronize()
                s = state.setdefault(mid, {'min_b': math.inf, 'max_a': -math.inf, 'fired': False})
                s['max_a'] = max(s['max_a'], torch.cuda.max_memory_allocated())
            return post
        handles.append(dmod.register_forward_pre_hook(mk_pre(id(dmod))))
        handles.append(dmod.register_forward_hook(mk_post(id(dmod))))

    peaks = []
    fired_count_per_iter = []
    # Try the most permissive kwargs first; fall back to bare input_ids
    kwarg_attempts = [
        {'input_ids': ids, 'attention_mask': mask},
        {'input_ids': ids},
    ]
    try:
        for i in range(repeats):
            for cn in list(state.keys()): state[cn] = {'min_b': math.inf, 'max_a': -math.inf, 'fired': False}
            reset_cuda()
            ok = False
            for kw in kwarg_attempts:
                try:
                    with torch.no_grad():
                        _ = model(**kw)
                    ok = True
                    break
                except (TypeError, RuntimeError) as e:
                    if i == 0:  # only print once
                        print(f'    [language] text-only forward with {list(kw)} failed: {type(e).__name__}: {str(e)[:80]}')
                    continue
                except torch.cuda.OutOfMemoryError:
                    print(f'    [language] text-only forward OOM')
                    return None, 0
            if not ok:
                continue
            torch.cuda.synchronize()
            # Aggregate envelopes across all hooked descendants
            descendant_peaks = [s['max_a'] - s['min_b'] for s in state.values() if s['fired'] and s['max_a'] > -math.inf]
            fired_count_per_iter.append(sum(1 for s in state.values() if s['fired']))
            if descendant_peaks:
                peaks.append(max(descendant_peaks) / 1024**2)
    finally:
        for h in handles:
            h.remove()

    median_peak = float(np.median(peaks)) if peaks else None
    median_fired = int(np.median(fired_count_per_iter)) if fired_count_per_iter else 0
    return median_peak, median_fired

print('v4 helpers ready: measure_vision_direct(), measure_language_textonly()')

v4 helpers ready: measure_vision_direct(), measure_language_textonly()


---
## 3. Molmo retry (v4)

In [4]:
molmo_spec = next(m for m in MODELS if m.name == 'Molmo-7B-D')
print(f'Auditing {molmo_spec.name} ({molmo_spec.hf_id}) — Phase-3 v4 hybrid')

molmo_result = {}
model = None
try:
    t0 = time.time()
    print('\nLoading Molmo (fp16, once)...')
    model = _load_model(molmo_spec, dtype=torch.float16, quantization=None, logger=None)
    model = model.to('cuda').eval()
    comps = extract_components(model, molmo_spec.family)
    print(f'  loaded + extracted in {time.time()-t0:.1f}s')
    comp_names = [n for n, c in comps.items() if c is not None]
    print(f'  components present: {comp_names}')

    # Vision via direct call
    print('\n--- Vision: direct call with kwarg sweep ---')
    if comps.get('vision') is not None:
        v_peak, v_pattern = measure_vision_direct(comps['vision'], image_hw=molmo_spec.native_image_hw, device='cuda', repeats=3)
        molmo_result['vision_mb'] = v_peak
        molmo_result['vision_pattern'] = v_pattern
        print(f'  vision peak_mem = {v_peak:.1f} MB (pattern={v_pattern})' if v_peak is not None else f'  vision FAILED (last pattern: {v_pattern})')

    # Language via text-only model forward + hooks
    print('\n--- Language: text-only model forward + descendant hooks ---')
    if comps.get('language') is not None:
        l_peak, l_fired = measure_language_textonly(model, comps['language'], text_tokens=32, device='cuda', repeats=3)
        molmo_result['language_mb'] = l_peak
        molmo_result['language_descendants_fired'] = l_fired
        print(f'  language peak_mem = {l_peak:.1f} MB ({l_fired} descendants fired)' if l_peak is not None else f'  language FAILED (fired={l_fired})')

    # Total = sum of populated components
    populated = [v for k, v in molmo_result.items() if k.endswith('_mb') and v is not None]
    if populated:
        molmo_result['total_mb'] = sum(populated)
        molmo_result['total_GB'] = round(sum(populated) / 1024, 3)
        molmo_result['success'] = True
        print(f'\n  TOTAL = {sum(populated):.1f} MB = {sum(populated)/1024:.2f} GB')
    else:
        molmo_result['success'] = False
        print(f'\n  All components failed.')
finally:
    if model is not None:
        del model
    reset_cuda()

molmo_df = pd.DataFrame([molmo_result])
molmo_df.to_csv(MEASUREMENT_DIR / 'nb21_molmo_retry.csv', index=False)
print(f'\nSaved nb21_molmo_retry.csv')
molmo_df

Auditing Molmo-7B-D (allenai/Molmo-7B-D-0924) — Phase-3 v4 hybrid

Loading Molmo (fp16, once)...


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

  loaded + extracted in 177.1s
  components present: ['vision', 'language']

--- Vision: direct call with kwarg sweep ---


ValueError: too many values to unpack (expected 4)

---
## 4. DeepSeek-VL2 retry (v4)

In [ ]:
dsvl2_spec = next(m for m in MODELS if m.name == 'DeepSeek-VL2-Small')
print(f'Auditing {dsvl2_spec.name} ({dsvl2_spec.hf_id}) — Phase-3 v4 hybrid')

DSVL2_CONFIGS = [
    {'name': 'std_fp16',     'quantization': None},
    {'name': 'int8_via_bnb', 'quantization': 'int8'},
]

dsvl2_results = []
for cfg in DSVL2_CONFIGS:
    print(f'\n--- Config: {cfg["name"]} ---')
    model = None
    row = {**cfg}
    try:
        t0 = time.time()
        model = _load_model(dsvl2_spec, dtype=torch.float16, quantization=cfg['quantization'], logger=None)
        if hasattr(model, 'eval'): model.eval()
        try:
            model = model.to('cuda')
        except Exception as e:
            print(f'  model.to(cuda) (continuing): {e}')
        comps = extract_components(model, dsvl2_spec.family)
        print(f'  loaded + extracted in {time.time()-t0:.1f}s')
        comp_names = [n for n, c in comps.items() if c is not None]
        print(f'  components present: {comp_names}')

        # Vision direct
        if comps.get('vision') is not None:
            v_peak, v_pattern = measure_vision_direct(comps['vision'], image_hw=(384, 384), device='cuda', repeats=3)
            row['vision_mb'] = v_peak
            row['vision_pattern'] = v_pattern
            print(f'  vision = {v_peak:.1f} MB (pattern={v_pattern})' if v_peak is not None else f'  vision FAILED')
        # Language text-only
        if comps.get('language') is not None:
            l_peak, l_fired = measure_language_textonly(model, comps['language'], text_tokens=32, device='cuda', repeats=3)
            row['language_mb'] = l_peak
            row['language_descendants_fired'] = l_fired
            print(f'  language = {l_peak:.1f} MB ({l_fired} descendants fired)' if l_peak is not None else f'  language FAILED')
        populated = [v for k, v in row.items() if k.endswith('_mb') and v is not None]
        if populated:
            row['total_mb'] = sum(populated)
            row['total_GB'] = round(sum(populated)/1024, 3)
            row['success'] = True
            print(f'  TOTAL = {sum(populated):.1f} MB')
        else:
            row['success'] = False
        dsvl2_results.append(row)
    except torch.cuda.OutOfMemoryError as e:
        print(f'  OOM on load: {str(e)[:120]}')
        dsvl2_results.append({**row, 'success': False, 'error': 'CUDA OOM on load'})
    except Exception as e:
        print(f'  FAILED: {type(e).__name__}: {str(e)[:180]}')
        dsvl2_results.append({**row, 'success': False, 'error': f'{type(e).__name__}: {str(e)[:180]}'})
    finally:
        if model is not None: del model
        reset_cuda()

dsvl2_df = pd.DataFrame(dsvl2_results)
dsvl2_df.to_csv(MEASUREMENT_DIR / 'nb21_dsvl2_retry.csv', index=False)
print(f'\nSaved nb21_dsvl2_retry.csv')
dsvl2_df

---
## 5. Update nb04_memory_summary.csv (additive, audit-preserving)

In [ ]:
mem_path = MEASUREMENT_DIR / 'nb04_memory_summary.csv'
if not mem_path.exists():
    print(f'WARN: {mem_path} not found')
else:
    mem_df = pd.read_csv(mem_path)
    if 'phase3_measurement_config' not in mem_df.columns:
        mem_df['phase3_measurement_config'] = ''

    def _apply(model_name, result_dict_or_list, label):
        if isinstance(result_dict_or_list, list):
            best = next((r for r in result_dict_or_list if r.get('success')), None)
        else:
            best = result_dict_or_list if result_dict_or_list.get('success') else None
        if not best:
            mem_df.loc[mem_df['name']==model_name, 'phase3_measurement_config'] = 'all_v4_configs_failed'
            print(f'  {label}: failed — annotated')
            return
        # apply per-component values
        any_pc = False
        for k in [k for k in best if k.endswith('_mb') and not k.startswith('total_')]:
            cname = k[:-3]
            val = best.get(k)
            if val is not None:
                col = f'{cname}_mem_mb'
                if col not in mem_df.columns: mem_df[col] = np.nan
                mem_df.loc[mem_df['name']==model_name, col] = val
                any_pc = True
        total = best.get('total_mb')
        if total is not None:
            mem_df.loc[mem_df['name']==model_name, 'total_mb'] = total
            mem_df.loc[mem_df['name']==model_name, 'total_GB'] = round(total/1024, 3)
        annotation_parts = ['v4_hybrid']
        if best.get('vision_pattern'): annotation_parts.append(f'vis={best["vision_pattern"]}')
        if 'name' in best and isinstance(best.get('name'), str): annotation_parts.append(f'load={best["name"]}')
        if not any_pc: annotation_parts.append('total_only')
        mem_df.loc[mem_df['name']==model_name, 'phase3_measurement_config'] = ' '.join(annotation_parts)
        print(f'  {label}: {" ".join(annotation_parts)}; total={total:.1f} MB' if total else f'  {label}: {" ".join(annotation_parts)}')

    _apply('Molmo-7B-D',         molmo_result, 'Molmo')
    _apply('DeepSeek-VL2-Small', dsvl2_results, 'DeepSeek-VL2')
    mem_df.to_csv(mem_path, index=False)
    print(f'\nSaved updated {mem_path}')
    mem_df[mem_df['name'].isin(['Molmo-7B-D','DeepSeek-VL2-Small'])][['name','total_mb','total_GB','phase3_measurement_config']]

---
## 6. Limitations note (paste into paper §8)

**Memory measurement caveats discovered in Phase-3 nb21 v4:**

* Molmo-7B-D and DeepSeek-VL2-Small required the v4 hybrid measurement protocol instead of the standard per-component direct call (v1) or unified model forward (v2/v3). The standard protocol fails because (a) Molmo's vision backbone uses `images=` instead of `pixel_values=` and accepts 5D `(B, num_crops, 3, H, W)` tensors, (b) Molmo's processor has a hard tensorflow dependency we did not install, (c) Molmo's `MolmoForCausalLM` wrapper also rejects `pixel_values=`, and (d) the language container in both models has no own `forward()` — the LLM body is invoked by the model's own implementation, not directly.
* The v4 hybrid protocol measures vision via a direct call on the vision backbone with a kwarg sweep (`pixel_values`/`images` × 4D/5D shapes); and measures language via a text-only `model(input_ids=...)` forward with descendant hooks on the LLM body.
* The `phase3_measurement_config` column records `v4_hybrid vis=<pattern> load=<config>` so reviewers can trace exactly which path produced each row's number.
* **Caveat for cross-row comparison**: the v4 hybrid numbers for these two rows reflect peak memory during a per-component forward, not during a unified multimodal forward. They are directly comparable to the per-component breakdowns in the other 18 rows but slightly under-represent the joint-forward peak that a real deployment would observe.

---
## 7. Saved outputs

* `outputs/measurements/nb21_molmo_retry.csv` — Molmo v4 result (1 row)
* `outputs/measurements/nb21_dsvl2_retry.csv` — DeepSeek-VL2 v4 results (2 configs)
* `outputs/measurements/nb04_memory_summary.csv` — Molmo + DeepSeek-VL2 rows patched (additive `phase3_measurement_config` = 'v4_hybrid ...')

**Status:** confirmatory if vision direct-call works for both models; partial-confirmatory if only language fires; honest-documentation if both fail.